In [ ]:
import liana as li
import omnipath as op
import decoupler as dc
import pandas as pd
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("/projects/CRC_xenium/xenium_data/merged_obj.h5ad")
coords_df = pd.read_csv("/projects/CRC_xenium/xenium_data/xenium_coords.csv", index_col=0)
coords_df = coords_df.reindex(adata.obs_names)
adata.obsm["spatial"] = coords_df[["x", "y"]].values

# check
print(adata)

In [ ]:
CN_info = pd.read_csv("/projects/CRC_xenium/xenium_data/CN_results/ALL_cells_r=50_k=20_CN=10.csv", index_col=0)
CN_info = CN_info.set_index('cell_id')
CN_info.index.name = None
adata.obs["All_CN_new"] = CN_info["neighborhood20"]

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Extract cell to cell interaction information
ligrec = op.interactions.import_intercell_network()
ligrec = ligrec.rename(columns={'genesymbol_intercell_source':'ligand', 'genesymbol_intercell_target':'receptor'})
ligrec = ligrec[['ligand', 'receptor', 'references'] + [col for col in ligrec.columns if col not in ['ligand', 'receptor', 'references']]]
ligrec = ligrec[ligrec["references"].str.contains("CellPhoneDB|CellChatDB|connectomeDB2020", na=False)]
ligrec = ligrec[ligrec["consensus_direction"] == True]

In [ ]:
# Filtering the data using CN information
adata_fil = adata[adata.obs["All_CN_new"]==5.0].copy()

In [ ]:
# Run rank_aggregate
li.mt.rank_aggregate(adata_fil, 
                     groupby='sub_type',
                     resource = ligrec[["ligand", 'receptor']],
                     n_jobs = 10,
                     expr_prop=0.05,
                     verbose=True,
                     use_raw = False)
res = adata_fil.uns['liana_res']

In [ ]:
z_scores = pd.read_csv("~/projects/CRC_xenium/xenium_data/CN_resluts/zscore_value_CN10.csv")

In [ ]:
cols_to_keep = z_scores.columns[z_scores.iloc[5] >= 0.5]
print(cols_to_keep)

In [ ]:
filtered_res = res[
    (res["specificity_rank"] < 0.05) & 
    (res["source"].isin(['CD8T_Eff_GNLY', 'IFITM3_Epithelial'])) &  
    (res["target"].isin(['CD8T_Eff_GNLY', 'IFITM3_Epithelial']))
]

In [ ]:
filtered_res.sort_values("magnitude_rank", ascending=True).head(50)

In [ ]:
# 1. Extract the original LIANA results
res = adata_fil.uns['liana_res'].copy()

# 2. Filter interactions by selected source and target cell types
cells = ['CD8T_Eff_GNLY', 'IFITM3_Epithelial']
res = res[res['source'].isin(cells) & res['target'].isin(cells)]
res = res[res['source'] != res['target']]

res = res[res["specificity_rank"]<0.05]

In [ ]:
res.to_csv("~/projects/CRC_xenium/xenium_data/CN5_celltocell.csv")